# 1. 주가 데이터 생성

In [37]:
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import random
from tqdm import tqdm

In [22]:
def generate_stock_data(n_days = 10, trend = 'up', volatility=1.0):
    start = datetime(2018, 1, 1)
    end = datetime(2023, 12 ,31)
    days_between = (end - start).days
    start_date = start + timedelta(days = np.random.randint(0, days_between))

    # 주말 제외
    dates = []
    current_date = start_date
    while len(dates) < n_days:
        if current_date.weekday() < 5: # 평일 이면
            dates.append(current_date)
        current_date += timedelta(days=1)

    # 가격 범위 스케일 랜덤 선택
    price_scale = np.random.choice([
        np.random.uniform(100, 1000), # 백단위
        np.random.uniform(1000, 10000), # 천단위
        np.random.uniform(10000, 100000), # 만단위
        np.random.uniform(100000, 1000000), # 십만 단위
    ])

    # 기본 방향 설정
    base_direction = 1 if trend == "up" else -1
    volatility = np.random.uniform(0.5, 2.0) # 변동성 랜덤
    
    data = []
    current_price = price_scale

    for date in dates:
        # 가격 변동 생성
        price_change = np.random.normal(base_direction * 0.02, volatility * 0.01)
        current_price *= (1 + price_change)

        # OHLC 데이터 생성
        daily_volatility = np.random.uniform(0.005, 0.03)
        open_price = current_price * (1 + np.random.normal(0, daily_volatility))
        high_price = max(open_price, current_price) * (1 + abs(np.random.normal(0, daily_volatility)))
        low_price = min(open_price, current_price) * (1 - abs(np.random.normal(0, daily_volatility)))
        close_price = current_price

        # inc- * 값 생성 (퍼센트 단위로)
        inc_values = []
        base_inc = np.random.normal(base_direction * 5, 3) # 기본 증감률
        
        for days in [5, 10, 15, 20, 25, 30]:
            if trend == 'up':
                inc = base_inc + np.random.normal(days/10, days/20)
            else:
                inc = base_inc - np.random.normal(days/10, days/20)
            inc_values.append(round(inc, 1))

        data.append({
            'date': date.strftime('%Y-%m-%d'),
            'open': round(open_price, 1),
            'high': round(high_price, 1),
            'low' : round(low_price, 1),
            'close': round(close_price, 1),
            'adj-close': round(close_price, 1),
            'inc-5': inc_values[0],
            'inc-10': inc_values[1],
            'inc-15': inc_values[2],
            'inc-20': inc_values[3],
            'inc-25': inc_values[4],
            'inc-30': inc_values[5],
        })

    df = pd.DataFrame(data)
    return df[['date', 'open', 'high', 'low', 'close', 'adj-close', 'inc-5', 'inc-10', 'inc-15', 'inc-20', 'inc-25', 'inc-30']]

generate_stock_data()

,date,open,high,low,close,adj-close,inc-5,inc-10,inc-15,inc-20,inc-25,inc-30
0,2019-04-22,47770.1,48808.5,47381.6,47453.8,47453.8,8.5,7.9,10.1,9.5,11.3,10.4
1,2019-04-23,47799.1,47873.6,47260.3,47359.4,47359.4,9.1,8.7,9.9,10.6,11.3,11.4
2,2019-04-24,47258.8,48689.0,44771.5,47568.0,47568.0,4.5,4.7,4.7,6.5,5.5,6.6
3,2019-04-25,47662.0,49100.9,47436.7,48464.8,48464.8,5.6,6.3,6.4,6.7,7.9,7.6
4,2019-04-26,48116.1,49631.1,47472.7,49192.2,49192.2,1.3,2.1,2.6,2.1,2.8,1.3
5,2019-04-29,49803.3,50394.7,49419.3,49534.7,49534.7,9.3,9.5,9.6,12.4,10.0,12.3
6,2019-04-30,50024.8,51072.7,49989.0,50770.3,50770.3,1.1,2.2,2.3,2.5,3.7,3.6
7,2019-05-01,51268.2,51922.3,50927.0,51349.8,51349.8,-1.2,0.2,-0.2,1.1,0.3,1.8
8,2019-05-02,54753.0,57039.7,50107.2,51728.1,51728.1,8.0,8.0,10.4,9.7,8.6,12.1
9,2019-05-03,52086.6,52689.0,51709.9,52565.0,52565.0,4.1,3.7,5.1,6.5,5.7,7.4


# 2. formatted dataset 생성

In [38]:
def df_to_markdown(df):
    markdown = """
|    |       date |      open |      high |       low |     close | adj-close | inc-5 | inc-10 | inc-15 | inc-20 | inc-25 | inc-30 |
|---:|:-----------|-------:|-------:|-------:|-------:|-------:|-------:|-------:|-------:|-------:|-------:|-------:|""" 

    for idx, row in df.iterrows():
        row = [f"  {idx}"] + list(row)
        markdown += ("\n|" + ' | '.join(list(map(str, row))) + " |")

    return markdown

In [40]:

stock_data = []
random.seed(42)
random_half = random.sample(range(5000), k=2500)

for idx in tqdm(range(5000)):
    if idx in random_half:
        trend = "up"
    else:
        trend = "down"

    new_df = generate_stock_data(trend = trend)
    markdown = df_to_markdown(new_df)

    formatted_text = f"""
표를 바탕으로 향후 주가가 상승할지 하락할지 예측하고 [증가] 혹은 [감소] 라고 답변하시요.

### 분석:{markdown}
    """
    answer = "[증가]" if trend == "up" else "[감소]"

    item = {"input_text" : formatted_text, "answer": answer}
    stock_data.append(item)
    
stock_data_df = pd.DataFrame(stock_data)
stock_data_df.to_csv('datasets/stock_prediction_dataset_2.csv', index=False)


100%|██████████| 5000/5000 [00:04<00:00, 1250.00it/s]
